# NewsNinja: AI News Digest Agent

Welcome to the interactive Google Colab notebook for the **NewsNinja** AI Agent!

This application uses an intelligent multi-step approach powered by **LangGraph**, where we define simple "nodes" representing different thinking or action phases:
1. **Parser**: Optimizes the user's raw topic.
2. **Searcher**: Uses Tavily web search to find current news articles.
3. **Writer**: Uses a Groq LLM to analyze the articles and write a digest.

Let's get started by installing the necessary dependencies.


In [ ]:
!pip install -qU langchain-groq langgraph langchain-community tavily-python

## Setup API Keys

We need to authenticate with `Groq` (for the LLM) and `Tavily` (for the search engine). When you run this cell, it will securely prompt you to enter both API keys.


In [ ]:
import os
import getpass

print("Enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass.getpass()

print("Enter your Tavily API Key:")
os.environ["TAVILY_API_KEY"] = getpass.getpass()

## Step 1: Define Tools

LangChain tools are functions the agent can use to interact with the outside world. Here, we define `search_news` using Tavily's search tool wrapper.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

@tool
def search_news(query: str) -> str:
    """Search the web for the latest news articles on a given topic, simply."""
    search = TavilySearchResults(
        max_results=5,
        search_depth="advanced",
        include_answer=True,
    )
    
    results = search.invoke({"query": query})
    
    if not results:
        return "No results found for the given query."
        
    formatted = []
    for i, result in enumerate(results, 1):
        title = result.get("title", "No Title")
        url = result.get("url", "")
        content = result.get("content", "No content available.")
        formatted.append(
            f"**Article {i}:** {title}\n"
            f"   URL: {url}\n"
            f"   Summary: {content}\n"
        )
        
    return "\n".join(formatted)

## Step 2: Define Agent Graph

Now we create our LangGraph structure. We define our `AgentState` which holds the variables passed between nodes, and then we define three functions representing the three nodes. Finally, we link them sequentially.


In [ ]:
from datetime import datetime
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# Define State
class AgentState(TypedDict):
    topic: str               # Raw user topic
    search_query: str        # Optimized search query
    search_results: str      # Raw search output
    digest: str              # Final polished news digest

# Node 1: Parser
def parse_topic(state: AgentState) -> dict:
    """Parse & refine user topic into an optimized search query."""
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    prompt = (
        "You are a search query optimizer. Given the following topic, create a concise, "
        "effective search query to find the latest news about it. "
        "Return ONLY the optimized search query, nothing else.\n\n"
        f"Topic: {state['topic']}"
    )
    response = llm.invoke(prompt)
    search_query = response.content.strip()
    print(f"\n[Node: parse_topic] Optimized search query: {search_query}")
    return {"search_query": search_query}

# Node 2: Searcher
def search_web(state: AgentState) -> dict:
    """Use Tavily to search for news articles."""
    query = state.get("search_query", state["topic"])
    print(f"\n[Node: search_web] Searching the web for: {query}")
    results = search_news.invoke({"query": query})
    print(f"Found {results.count('Article')} articles")
    return {"search_results": results}

# Node 3: Writer
def write_digest(state: AgentState) -> dict:
    """Use Groq LLM to synthesize a polished news digest."""
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    prompt = (
        "You are NewsNinja, an elite AI news analyst. Based on the following search "
        "results, create a comprehensive yet concise news digest.\n\n"
        "FORMAT YOUR RESPONSE AS:\n"
        "**NewsNinja Digest**\n"
        f"**Topic:** {state['topic']}\n"
        f"**Date:** {datetime.now().strftime('%B %d, %Y')}\n\n"
        "---\n\n"
        "Then provide:\n"
        "1. A brief **Executive Summary** (2-3 sentences)\n"
        "2. **Key Headlines** with bullet points\n"
        "3. **Analysis & Insights** (what this means)\n"
        "4. **Sources** (list the URLs)\n\n"
        "---\n\n"
        f"SEARCH RESULTS:\n{state['search_results']}"
    )
    response = llm.invoke(prompt)
    print("\n[Node: write_digest] Digest written!")
    return {"digest": response.content}

# Compile Graph
def build_graph() -> StateGraph:
    graph = StateGraph(AgentState)
    graph.add_node("parser", parse_topic)
    graph.add_node("searcher", search_web)
    graph.add_node("writer", write_digest)
    
    graph.set_entry_point("parser")
    graph.add_edge("parser", "searcher")
    graph.add_edge("searcher", "writer")
    graph.add_edge("writer", END)
    
    return graph.compile()

# Build it
app = build_graph()

## Step 3: Run NewsNinja

Run the cell below, enter your topic, and watch the agent create your personalized news digest!


In [ ]:
topic = input("Enter a news topic to research: ")

if topic.strip():
    print("=" * 60)
    print(f"Starting pipeline for: {topic}")
    print("-" * 60)

    # Invoke the graph
    result = app.invoke({"topic": topic})

    print("\n" + "=" * 60)
    print(result["digest"])
    print("=" * 60)
else:
    print("No topic provided. Please try again.")

## Conclusion

You've successfully run an Agentic workflow! The LangGraph structure allowed the system to sequentially refine the query, find the information, and then intelligently summarize it. You can modify the nodes above or add new nodes (e.g. `fact_checker` or `slack_publisher`) to expand the graph.
